In [3]:
import sys
print(sys.version)

3.12.5 (tags/v3.12.5:ff3bc82, Aug  6 2024, 20:45:27) [MSC v.1940 64 bit (AMD64)]


# Phase 1 — Domain & Data Collection

## Project Domain

The domain of this RAG assistant is AI, Machine Learning, and Deep Learning
technical documentation and educational resources.

The assistant will allow users to ask questions about topics related to
Python, Machine Learning, Deep Learning, TensorFlow, scikit-learn, and
PyTorch, and retrieve relevant information from the collected documents.

## Data Sources

The document collection consists of PDF-based technical and educational
resources stored in the `data/documents/` directory.

The documents are organized into folders where necessary to keep the
document collection structured.

# RAG-Powered Document Assistant

## 1. Load & Inspect Documents

In this section, we load the source PDF documents and inspect their
extracted text before preparing them for the RAG pipeline.

In [4]:
from pathlib import Path
from pypdf import PdfReader

In [5]:
#adding the path of the data 

documents_path = Path("../docs-pdf")

print("Documents folder:", documents_path)
print("Folder exists:", documents_path.exists())

Documents folder: ..\docs-pdf
Folder exists: True


In [6]:
# make sure that all of the data is inserted in the code 

pdf_files = list(documents_path.rglob("*.pdf"))

print("Number of PDF files:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)

Number of PDF files: 29
- c-api.pdf
- distributing.pdf
- extending.pdf
- faq.pdf
- howto-annotations.pdf
- howto-argparse.pdf
- howto-clinic.pdf
- howto-cporting.pdf
- howto-curses.pdf
- howto-descriptor.pdf
- howto-functional.pdf
- howto-instrumentation.pdf
- howto-ipaddress.pdf
- howto-logging-cookbook.pdf
- howto-logging.pdf
- howto-pyporting.pdf
- howto-regex.pdf
- howto-sockets.pdf
- howto-sorting.pdf
- howto-unicode.pdf
- howto-urllib2.pdf
- installing.pdf
- library.pdf
- reference.pdf
- scikit-learn-docs.pdf
- TensorFlow-User-Guide.pdf
- tutorial.pdf
- using.pdf
- whatsnew.pdf


In [7]:
# open the PDFs and and extarct text to load in the next cell using the function load_pdf

def load_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    
    text = ""
    
    for page in reader.pages: #function to keep opening pages in PDFs
        page_text = page.extract_text()
        
        if page_text:
            text += page_text + "\n"
    
    return text

In [8]:
#reads the content of every PDF

documents = {} # to store the extracted text in the empty dictionary

for pdf in pdf_files:
    text = load_pdf(pdf) 
    documents[str(pdf.relative_to(documents_path))] = text # actual storing 
    
    print(f"✅ {pdf.relative_to(documents_path)}: {len(text):,} characters")

✅ c-api.pdf: 795,681 characters
✅ distributing.pdf: 107,771 characters
✅ extending.pdf: 256,807 characters
✅ faq.pdf: 269,684 characters
✅ howto-annotations.pdf: 8,840 characters
✅ howto-argparse.pdf: 20,096 characters
✅ howto-clinic.pdf: 58,554 characters
✅ howto-cporting.pdf: 778 characters
✅ howto-curses.pdf: 21,790 characters
✅ howto-descriptor.pdf: 38,395 characters
✅ howto-functional.pdf: 45,814 characters
✅ howto-instrumentation.pdf: 12,796 characters
✅ howto-ipaddress.pdf: 11,480 characters
✅ howto-logging-cookbook.pdf: 127,163 characters
✅ howto-logging.pdf: 39,974 characters
✅ howto-pyporting.pdf: 21,300 characters
✅ howto-regex.pdf: 50,248 characters
✅ howto-sockets.pdf: 18,198 characters
✅ howto-sorting.pdf: 10,701 characters
✅ howto-unicode.pdf: 27,808 characters
✅ howto-urllib2.pdf: 22,881 characters
✅ installing.pdf: 109,956 characters
✅ library.pdf: 5,375,345 characters
✅ reference.pdf: 460,466 characters
✅ scikit-learn-docs.pdf: 5,119,909 characters
✅ TensorFlow-User-G

In [9]:
#making sire that all the text in the pdfs are extarcted 

for filename, text in documents.items(): # gives ther file name and the text
    if len(text.strip()) > 100:
        print(f"✅ {filename} - Text extraction successful")
    else:
        print(f"⚠️ {filename} - Very little text extracted; OCR may be needed")

✅ c-api.pdf - Text extraction successful
✅ distributing.pdf - Text extraction successful
✅ extending.pdf - Text extraction successful
✅ faq.pdf - Text extraction successful
✅ howto-annotations.pdf - Text extraction successful
✅ howto-argparse.pdf - Text extraction successful
✅ howto-clinic.pdf - Text extraction successful
✅ howto-cporting.pdf - Text extraction successful
✅ howto-curses.pdf - Text extraction successful
✅ howto-descriptor.pdf - Text extraction successful
✅ howto-functional.pdf - Text extraction successful
✅ howto-instrumentation.pdf - Text extraction successful
✅ howto-ipaddress.pdf - Text extraction successful
✅ howto-logging-cookbook.pdf - Text extraction successful
✅ howto-logging.pdf - Text extraction successful
✅ howto-pyporting.pdf - Text extraction successful
✅ howto-regex.pdf - Text extraction successful
✅ howto-sockets.pdf - Text extraction successful
✅ howto-sorting.pdf - Text extraction successful
✅ howto-unicode.pdf - Text extraction successful
✅ howto-urllib

## 2.2 Chunking Strategy

We split the extracted documents into fixed-size chunks of approximately
1000 characters with an overlap of 200 characters.

We use fixed-size chunking with 1000 characters per chunk and 200 characters of overlap, which helps preserve context across chunk boundaries while keeping the retrieved information focused.

An overlap of 200 means that we will retake the last 2 character from the last chunck

Overlapping is important because sometimes the boundries are important

In [10]:
#creating the chunck with a fixed chunk size and overlap 

def create_chunks(text, chunk_size=1000, overlap=200):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [11]:
chunks = [] #contains all the chunks of the pdfs

for filename, text in documents.items(): #Go through every PDF and get its filename and extracted text.

    document_chunks = create_chunks( # chunk of one document uding function create_chunk in the cell before that one 
        text,
        chunk_size=1000,
        overlap=200
    )

    for i, chunk in enumerate(document_chunks): #gets the index and the content of the chunk

        chunks.append({
            "text": chunk,
            "source": filename,
            "chunk_id": i
        })

print("Total chunks:", len(chunks))

Total chunks: 17162


## 2.3 Embeddings

Each document chunk is converted into a numerical vector called an embedding.
We use the `all-MiniLM-L6-v2` model from Sentence Transformers which switch text to a numeric vectors to make searching easier for the RAG

In [12]:
#importing pretrained models for embedding 

from sentence_transformers import SentenceTransformer # needed to get pretrained models used for embedding

embedding_model = SentenceTransformer("all-MiniLM-L6-v2") # loaded th embedding model into a variable

test_embedding = embedding_model.encode("What is overfitting?") # convert question to embedding 

print("Embedding dimensions of each chunk:", len(test_embedding))

c:\Users\Youssef Elshamandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3226.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimensions of each chunk: 384


In [13]:
#takes all the texts and convert them into embeddings 

texts = [chunk["text"] for chunk in chunks] # get the text of every chunk, related to cell 9

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Number of chunks:", len(texts))
print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", len(embeddings[0]))

Batches: 100%|██████████| 537/537 [37:10<00:00,  4.15s/it]


Number of chunks: 17162
Number of embeddings: 17162
Embedding dimensions: 384


## 2.4 Vector Store — ChromaDB

The generated embeddings will be stored in ChromaDB together with the original
text chunks and their source information.

Embedding → to find 
Text → to answer 
Source → where from 

In [ ]:
import chromadb # search the embedding for the similar chunk, chromadb stores text,embeddings, and source

vector_store_path = "../data/vector_store" # Save my ChromaDB data in the data/vector_store folder.

client = chromadb.PersistentClient( # create the connection to chromadb
    path=vector_store_path
)

print("ChromaDB client created successfully.")


ChromaDB client created successfully.


In [16]:
collection = client.get_or_create_collection( # create a folder inside chromadb called "rag_documents"
    name="rag_documents"
)

print("Collection created successfully.")

Collection created successfully.


In [18]:


client.delete_collection("rag_documents") # to make sure if er run the notebook again we do not create a new collection 

collection = client.get_or_create_collection(
    name="rag_documents"
)
# create id for every chunk, so chromadb knows which piece of info is which 
ids = [
    f"{chunk['source']}_{chunk['chunk_id']}"
    for chunk in chunks
]
# stores the info of every chunk, helps to tell where did the chunk come from
metadatas = [
    {
        "source": chunk["source"],
        "chunk_id": chunk["chunk_id"]
    }
    for chunk in chunks
]

# ChromaDB has a maximum batch size, so we add the data in smaller batches.
batch_size = 5000 # number of chunks per batch as chromadb can not get all chunks at once 

for start in range(0, len(chunks), batch_size):
    end = start + batch_size
# sets the collection with info it needs
    collection.add(
        ids=ids[start:end],
        documents=texts[start:end],
        embeddings=embeddings[start:end].tolist(), #converts the embeddings into a regular format that ChromaDB accepts,
        metadatas=metadatas[start:end]
    )

    print(f"Stored {min(end, len(chunks))} / {len(chunks)} chunks")

print("Stored chunks:", collection.count())

Stored 5000 / 17162 chunks
Stored 10000 / 17162 chunks
Stored 15000 / 17162 chunks
Stored 17162 / 17162 chunks
Stored chunks: 17162


In [19]:
def retrieve_documents(question, n_results=5):
 # first did the embedding then converted to a formate to be understood by chromadb  
    question_embedding = embedding_model.encode(question).tolist() 
# search the chromadb with the most relevant info in only 5 chunks
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    return results

In [ ]:
#testing cell
question = "What is overfitting?"

results = retrieve_documents(question, n_results=5)

for i, document in enumerate(results["documents"][0]):
    print(f"\n--- Result {i + 1} ---")
    print("Source:", results["metadatas"][0][i]["source"])
    print("Chunk ID:", results["metadatas"][0][i]["chunk_id"])
    print(document[:800])


--- Result 1 ---
Source: Deep-Learning-with-PyTorch.pdf
Chunk ID: 939
pos
Figure 12.18 Our positive loss showing clear signs of overfitting, as the training loss and 
validation loss are trending in different directions
345Revisiting the problem of overfitting
12.5 Revisiting the problem of overfitting
We touched on the concept of overfitting in  chapter 5, and now it’s time to take a
closer look at how to address this common situation. Our goal with training a model is
to teach it to recognize the general properties  of the classes we are interested in, as
expressed in our dataset. Those general prop erties are present in some or all samples
of the class and can be generalized and used to predict samples that haven’t been
trained on. When the model starts to learn specific properties of the training set, overfit-
ting occurs, and the model starts to lose th

--- Result 2 ---
Source: Deep-Learning-with-PyTorch.pdf
Chunk ID: 387
3, just because it pushes
the loss very close to zero. Si

In [20]:
#builds the instructions we will send to the LLM.

def build_prompt(question, results):
    context_parts = [] # empty list were the relevent chunks to a certain question will be stored

# goes through the 5 chunks retrieved by ChromaDB, one at a time.
    for i, document in enumerate(results["documents"][0]):
        source = results["metadatas"][0][i]["source"]

        context_parts.append(
            f"[Source {i + 1}: {source}]\n{document}"
        )

    context = "\n\n".join(context_parts) # combine chunks
# instructions sent to the LLM
    prompt = f"""
You are a helpful AI and Machine Learning document assistant.

Answer the user's question using ONLY the information provided in the context.

Rules:
- Do not use outside knowledge.
- Do not invent facts.
- If the answer is not available in the context, say:
  "The information was not found in the provided documents."
- At the end, mention the source document(s) used.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [ ]:
# testing cell
question = "What is overfitting?"

results = retrieve_documents(question, n_results=5)

prompt = build_prompt(question, results)

print(prompt[:5000])


You are a helpful AI and Machine Learning document assistant.

Answer the user's question using ONLY the information provided in the context.

Rules:
- Do not use outside knowledge.
- Do not invent facts.
- If the answer is not available in the context, say:
  "The information was not found in the provided documents."
- At the end, mention the source document(s) used.

Context:
[Source 1: Deep-Learning-with-PyTorch.pdf]
pos
Figure 12.18 Our positive loss showing clear signs of overfitting, as the training loss and 
validation loss are trending in different directions
345Revisiting the problem of overfitting
12.5 Revisiting the problem of overfitting
We touched on the concept of overfitting in  chapter 5, and now it’s time to take a
closer look at how to address this common situation. Our goal with training a model is
to teach it to recognize the general properties  of the classes we are interested in, as
expressed in our dataset. Those general prop erties are present in some or all sa

Setting the LLM

In [25]:
import ollama # tools that help you run the LLM on your computer 

ollama_model = "llama3.2"

print("Ollama model:", ollama_model)


Ollama model: llama3.2


In [26]:
response = ollama.chat(
    model="llama3.2", # the actual LLM
    messages=[
        {
            "role": "user", # message is coming from the user 
            "content": "What is overfitting?" # the question we are receiving 
        }
    ]
)

print(response["message"]["content"])

Overfitting is a phenomenon in machine learning where a model becomes too specialized to the training data and fails to generalize well to new, unseen data. In other words, a model that is too complex or has too many parameters can fit the noise and random fluctuations in the training data, resulting in poor performance on unseen data.

When a model is overfitting, it tends to:

1. Perform well on the training data, but poorly on the test data.
2. Have a high error rate on the test data.
3. Be sensitive to changes in the training data, such as small changes in the data distribution.

Overfitting occurs when a model has too many parameters, or when the model is too complex, allowing it to fit the noise in the training data. This can be mitigated by:

1. Regularization techniques, such as L1 or L2 regularization, dropout, or early stopping.
2. Data augmentation, such as adding noise or transforming the data.
3. Cross-validation, which helps evaluate the model's performance on unseen data

In [27]:
# The Main RAG function

def generate_answer(question, n_results=5):
    # Retrieve relevant documents
    results = retrieve_documents(
        question,
        n_results=n_results
    )

    # Build grounded prompt
    prompt = build_prompt(
        question,
        results
    )

    # Ask the local LLM
    response = ollama.chat(
        model=ollama_model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
# Getting the actual answer
    answer = response["message"]["content"]

    return answer, results

In [28]:
# testing cell
question = "What is overfitting?"

answer, results = generate_answer(
    question,
    n_results=5
)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

print("\nRetrieved Sources:")

for metadata in results["metadatas"][0]:
    print("-", metadata["source"])

Question:
What is overfitting?

Answer:
Overfitting is when a model is too complex and learns the noise in the training data, rather than generalizing well to new, unseen data.

Source document(s) used: scikit-learn-docs.pdf

Retrieved Sources:
- scikit-learn-docs.pdf
- scikit-learn-docs.pdf
- scikit-learn-docs.pdf
- scikit-learn-docs.pdf
- scikit-learn-docs.pdf


In [23]:
evaluation_questions = [
    "What is overfitting?",
    "What is underfitting?",
    "What is a neural network?",
    "What is a tensor?",
    "What is gradient descent?",
    "What is cross-validation?",
    "What is a decision tree?",
    "What is a convolutional neural network?",
    "What is an embedding?",
    "What is regularization?"
]

print("Number of evaluation questions:", len(evaluation_questions))

Number of evaluation questions: 10


In [29]:
evaluation_results = []

for i, question in enumerate(evaluation_questions, start=1):

    print(f"Processing question {i}/10...")

    answer, results = generate_answer(
        question,
        n_results=5
    )

    sources = [
        metadata["source"]
        for metadata in results["metadatas"][0]
    ]

    evaluation_results.append({
        "question": question,
        "sources": sources,
        "answer": answer,
        "correct": "",
        "grounded": ""
    })

print("\nEvaluation completed.")

Processing question 1/10...
Processing question 2/10...
Processing question 3/10...
Processing question 4/10...
Processing question 5/10...
Processing question 6/10...
Processing question 7/10...
Processing question 8/10...
Processing question 9/10...
Processing question 10/10...

Evaluation completed.


In [30]:
for i, result in enumerate(evaluation_results, start=1):

    print("\n" + "=" * 80)
    print(f"QUESTION {i}")
    print("=" * 80)

    print("Question:")
    print(result["question"])

    print("\nAnswer:")
    print(result["answer"])

    print("\nSources:")
    for source in result["sources"]:
        print("-", source)


QUESTION 1
Question:
What is overfitting?

Answer:
Overfitting is when a model is too complex and fits the training data too closely, resulting in poor performance on new, unseen data.

Source document: scikit-learn-docs.pdf (Rules 4.4. Inspection, 4.4.4. Limitations of impurity-based feature importances)

Sources:
- scikit-learn-docs.pdf
- scikit-learn-docs.pdf
- scikit-learn-docs.pdf
- scikit-learn-docs.pdf
- scikit-learn-docs.pdf

QUESTION 2
Question:
What is underfitting?

Answer:
The information was not found in the provided documents.

Source documents used: scikit-learn-docs.pdf

Sources:
- scikit-learn-docs.pdf
- scikit-learn-docs.pdf
- scikit-learn-docs.pdf
- scikit-learn-docs.pdf
- scikit-learn-docs.pdf

QUESTION 3
Question:
What is a neural network?

Answer:
A neural network is a supervised learning algorithm that learns a function f(·) :Rm→Ro by training on a dataset, where m is the number of dimensions for input and o is the number of dimensions for output.

Source docume

In [31]:
import pandas as pd

evaluation_table = pd.DataFrame(evaluation_results)

evaluation_table

,question,sources,answer,correct,grounded
0,What is overfitting?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",Overfitting is when a model is too complex and...,,
1,What is underfitting?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",The information was not found in the provided ...,,
2,What is a neural network?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",A neural network is a supervised learning algo...,,
3,What is a tensor?,"[TensorFlow-User-Guide.pdf, TensorFlow-User-Gu...",The information was not found in the provided ...,,
4,What is gradient descent?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",Gradient descent is an optimization method for...,,
5,What is cross-validation?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",Cross-validation is a procedure used to evalua...,,
6,What is a decision tree?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",A decision tree is a non-parametric supervised...,,
7,What is a convolutional neural network?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",The information was not found in the provided ...,,
8,What is an embedding?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...","According to Source 2: scikit-learn-docs.pdf, ...",,
9,What is regularization?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",Regularization is a penalty added to the loss ...,,


In [32]:
evaluation_table["correct"] = [
    "Yes",  # 1. Overfitting
    "No",   # 2. Underfitting
    "Yes",  # 3. Neural network
    "Yes",  # 4. Tensor
    "Yes",  # 5. Gradient descent
    "Yes",  # 6. Cross-validation
    "Yes",  # 7. Decision tree
    "Yes",  # 8. Convolutional neural network
    "Yes",  # 9. Embedding
    "Yes"   # 10. Regularization
]

evaluation_table["grounded"] = [
    "Yes",  # 1. Overfitting
    "No",   # 2. Underfitting
    "Yes",  # 3. Neural network
    "Yes",  # 4. Tensor
    "Yes",  # 5. Gradient descent
    "Yes",  # 6. Cross-validation
    "Yes",  # 7. Decision tree
    "Yes",  # 8. Convolutional neural network
    "Yes",  # 9. Embedding
    "Yes"   # 10. Regularization
]

evaluation_table

,question,sources,answer,correct,grounded
0,What is overfitting?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",Overfitting is when a model is too complex and...,Yes,Yes
1,What is underfitting?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",The information was not found in the provided ...,No,No
2,What is a neural network?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",A neural network is a supervised learning algo...,Yes,Yes
3,What is a tensor?,"[TensorFlow-User-Guide.pdf, TensorFlow-User-Gu...",The information was not found in the provided ...,Yes,Yes
4,What is gradient descent?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",Gradient descent is an optimization method for...,Yes,Yes
5,What is cross-validation?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",Cross-validation is a procedure used to evalua...,Yes,Yes
6,What is a decision tree?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",A decision tree is a non-parametric supervised...,Yes,Yes
7,What is a convolutional neural network?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",The information was not found in the provided ...,Yes,Yes
8,What is an embedding?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...","According to Source 2: scikit-learn-docs.pdf, ...",Yes,Yes
9,What is regularization?,"[scikit-learn-docs.pdf, scikit-learn-docs.pdf,...",Regularization is a penalty added to the loss ...,Yes,Yes


In [33]:
print("Total questions:", len(evaluation_table))

print(
    "Correct answers:",
    (evaluation_table["correct"] == "Yes").sum()
)

print(
    "Grounded answers:",
    (evaluation_table["grounded"] == "Yes").sum()
)

Total questions: 10
Correct answers: 9
Grounded answers: 9


In [34]:
failure_cases = []

for _, row in evaluation_table.iterrows():

    if row["correct"] != "Yes" or row["grounded"] != "Yes":

        failure_cases.append({
            "question": row["question"],
            "answer": row["answer"],
            "possible_mitigation": (
                "Improve chunking, retrieve more relevant chunks, "
                "or improve the RAG prompt."
            )
        })

print("Number of failure cases:", len(failure_cases))

for case in failure_cases:
    print("\nQuestion:", case["question"])
    print("Possible mitigation:", case["possible_mitigation"])

Number of failure cases: 1

Question: What is underfitting?
Possible mitigation: Improve chunking, retrieve more relevant chunks, or improve the RAG prompt.


In [35]:
print("Vector store path:")
print(vector_store_path)

print("\nCollection name:")
print(collection.name)

print("\nNumber of stored chunks:")
print(collection.count())

print("\nEmbedding model:")
print("all-MiniLM-L6-v2")

print("\nChunk size:")
print(1000)

print("\nChunk overlap:")
print(200)

print("\nRetrieval count:")
print(5)

Vector store path:
../data/vector_store

Collection name:
rag_documents

Number of stored chunks:
17162

Embedding model:
all-MiniLM-L6-v2

Chunk size:
1000

Chunk overlap:
200

Retrieval count:
5
